# TODO Basic Excel Extraction Example

This is an example of how a user would perform the following steps:
- Update the user's function access to include the Excel module
- Import the istari-digital-client from PyPI
- Instantiate an instance of the istari-digital-client Client class
- Upload a Microsoft Excel file
- Extract the Excel model
- View the extracted artifacts

## Update function access

The Istari administrator must add the user to the relevant function on the Function Access page. This can be accessed in the Istari environment via Admin Panel -> Function Access -> Relevant Function -> Manage Function Access. Search to find the appropriate user, click add at the top of the window, then click save at the bottom of the window. 

## Install dependencies from PyPI

#### Note: the pip command below installs the most recent version of the digital client. Depending on your Istari release, you may need to install an older version. Refer to your relevant release page in the [release docs](https://docs.istaridigital.com/releases/2025-06-01-Release) for the appropriate SDK client version. The commented line below shows the command to install a specific client version. 

In [2]:
!pip install istari-digital-client
# !pip install istari-digital-client==7.4.4 # use this command to install the appropriate client version if you are on an older Istari release 

!pip install python-dotenv

In [3]:
import istari_digital_client as istari_digital
import os
from dotenv import load_dotenv
from pathlib import Path
import time
from datetime import datetime

## Instantiate the istari-digital-client Client class

To interact with Istari Digital, we need to create an instance of the istari-digital-client Client class.

The Client class takes a Configuration object that contains the following parameters:
- registry_url (required): The URL of the Istari Digital Registry Service
- registry_auth_token (required): The authentication token to use to authenticate with the Istari Digital Registry Service
- retry_enabled (optional): Whether to retry failed requests.  Defaults to True
- retry_max_attempts (optional): The maximum number of retry attempts.  Defaults to 3
- retry_min_interval_millis (optional): The minimum interval between retry attempts in milliseconds.
- retry_max_interval_millis (optional): The maximum interval between retry attempts in milliseconds.
- retry_jitter_enabled (optional): Whether to use jitter when retrying failed requests.  Defaults to True
- filesystem_cache_enabled (optional): Whether to use the filesystem cache.  Defaults to True
- filesystem_cache_root (optional): The root directory of the filesystem cache.
- filesystem_cache_clean_on_exit (optional): Whether to clean the filesystem cache on exit.  Defaults to True
- multipart_chunksize (optional): The chunk size to use when uploading files.
- multipart_threshold (optional): The threshold size to use when uploading files.

We MUST set the registry_url and registry_auth_token parameters to interact with Istari Digital.
In this example, we are following best practices and using environment variables to store the
registry_url and registry_auth_token. The environment variables are loaded using the python-dotenv package.

The configuration parameters are then used to instantiate the Configuration class.
Once instantiated, the configuration object is passed to the istari-digital-client Client class to create
an instance of the Client class.

In [4]:
load_dotenv()

dev_auth_token = os.getenv("REGISTRY_ACCESS_TOKEN")
assert dev_auth_token is not None

dev_registry_url = os.getenv("REGISTRY_URL")
assert dev_registry_url is not None

configuration = istari_digital.Configuration(
    registry_url=dev_registry_url,
    registry_auth_token=dev_auth_token,
)
assert configuration is not None

client = istari_digital.Client(
    config = configuration
)
assert client is not None

2025-07-25 15:35:59 - istari-digital-client - INFO - Logging configured with level: INFO


## Upload the Excel file to the Istari Digital Registry Service

Before extracting the Excel file, the file must be uploaded to the Istari Digital Registry Service.
To do this, we use the add_model method of the client object.
The add_model method takes the following parameters:
- path: The path to the file to upload.
- description: An optional description of the file
- version_name: An optional version name of the file
- external_identifier: An optional external identifier of the file
- display_name: An optional display name of the file.  Useful for displaying in the UI

The parameters are then passed to the add_model method of the client object to upload the file.
The add_model method returns a Model object that contains the metadata of the uploaded file.

We can then validate that the file was successfully uploaded by accessing the properties of
the Model object that is returned.

In [5]:
base_path = Path.cwd() 
file_path = base_path / "files/Excel-test-Large.xlsx"
external_identifier= "1.0.0"
display_name="Excel Test Large v1"
description="Excel File Modification Demo"
version_name = "v1"

print(f"Upload File Path: {file_path}\n")

excel_model = client.add_model(
    path=file_path,
    version_name=version_name,
    external_identifier=external_identifier,
    display_name=display_name,
    description=description
)
print(f"Uploaded base model with ID {excel_model.id}")

assert excel_model is not None
assert isinstance(excel_model, istari_digital.Model)

assert excel_model.description == description
assert excel_model.version_name == version_name
assert excel_model.external_identifier == external_identifier
assert excel_model.display_name == display_name
assert excel_model.read_bytes() == file_path.read_bytes()

Upload File Path: /Users/matthewmiller/Projects/istari-digital-client-cookbook/integrations/files/Excel-test-Large.xlsx

Uploaded base model with ID 8e228b08-ae30-44a6-92ac-d3013b380dde


## TODO Extract the uploaded Excel file

To extract the uploaded Excel file, the following params are needed:
- model_id: The id of the model to extract
- function: The extraction function to run
- tool_name: The tool that is needed to extract the Catia file
- tool_version: The version of the tool
- operating_system: The operating system that the tool will run on

The extract parameters are then passed to the add_job method of the client object to begin the extraction job.

Once the job is created, we poll the job until it is completed.  The job is polled by checking the JobStatusName of the job.

In [14]:
functions = client.list_functions(
    tool_name="microsoft_office_excel",
    tool_version="2021",
    operating_system="Windows 11",
    #model_extension="xlsx"
)
for fn in functions:
    print(f"- {fn.name} (version: {fn.version})")

ValidationError: 1 validation error for ClientApi.list_functions
tool_name
  Unexpected keyword argument [type=unexpected_keyword_argument, input_value='microsoft_office_excel', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/unexpected_keyword_argument

In [ ]:
model_id = excel_model.id
function = "@istari:update_cell"
tool_name = "microsoft_office_excel"
tool_version = "2021"
operating_system = "Windows 11"

sheet_name = "test"
row = 4
column = 5 #note that column letters must be converted to number
new_cell_value = "999999"

job = client.add_job(
    model_id=model_id,
    function=function,
    tool_name=tool_name,
    tool_version=tool_version,
    operating_system=operating_system,
    sheet_name = sheet_name,
    row = row,
    column = column,
    new_cell_value = new_cell_value,
) 

job = client.add_job(
    model_id=model_id,
    function=function,
    tool_name=tool_name,
    tool_version=tool_version,
    operating_system=operating_system,
    sheet_name = sheet_name,
    row = row,
    column = 6,
    new_cell_value = new_cell_value,
) 

#error -- batch_execute not found for excel
""" job = client.add_job(
  model_id = excel_model.id,
  function = "@istari:batch_execute",
  tool_name = "microsoft_office_excel",
  tool_version = "2021",
  operating_system = "Windows 11",
  parameters = {
  "func_list": [
    {
      "name": "@istari:update_cell",
      "input": {
        "parameters": {
          "sheet_name": sheet_name,
          "row": row,
          "column": column,
          "new_cell_value": new_cell_value
        }
      }
    },
    {
      "name": "@istari:extract",
      "input": {}
    }
  ]
}
) """

start_time = datetime.now()
print(f"Extraction started for model ID {model_id}, job ID: {job.id}")

assert job is not None
assert isinstance(job, istari_digital.Job)

while job.status.name not in [istari_digital.JobStatusName.COMPLETED, istari_digital.JobStatusName.FAILED]:
    elapsed_time = datetime.now()-start_time
    print(f"\rModification job {job.id} status: {job.status.name.value} ({elapsed_time})", end="")
    time.sleep(5)
    job = client.get_job(job.id)

if job.status.name.value == "Completed":
    print(f"\r\rExtraction job {job.id} completed successfully!                         ")
    model = client.get_model(job.model.id)
    print(f"\nThe following artifacts were extracted:")
    for artifact in model.artifacts:
        print(f"- artifact id: {artifact.id}")
        print(f"  revision id: {artifact.revision.id}")
        print(f"  extension: {artifact.extension}")
        print(f"  mime type: {artifact.mime}")
        print(f"  name: {artifact.name}")
else:
    print(f"\r\rExtraction job {job.id} failed with status: {job.status.name.value}")

assert job.status.name == istari_digital.JobStatusName.COMPLETED

Extraction started for model ID 8e228b08-ae30-44a6-92ac-d3013b380dde, job ID: 7dc0d5ed-a015-430a-8e51-d6bc7e503d32
Extraction job 7dc0d5ed-a015-430a-8e51-d6bc7e503d32 completed successfully!                         ob 7dc0d5ed-a015-430a-8e51-d6bc7e503d32 status: Pending (0:00:05.638472)Modification job 7dc0d5ed-a015-430a-8e51-d6bc7e503d32 status: Running (0:00:11.258225)

The following artifacts were extracted:
- artifact id: bfabaf3c-6e02-4b0c-b9df-64084e0b0b6d
  revision id: 11fb234e-143d-456a-bcee-84028aea5bce
  extension: xlsx
  mime type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
  name: modified_workbook.xlsx
- artifact id: 2b5626bf-0605-4c62-9104-434ed945a6b1
  revision id: 21117843-f9ae-4da3-b70c-bf4cc16c8f48
  extension: xlsx
  mime type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
  name: modified_workbook.xlsx
- artifact id: d881bda9-665d-4ac3-a98e-47773db3b925
  revision id: ac900fc2-3802-4a83-8edf-68974a11c8ee
  extension: xlsx

## Clean up the uploaded file

The uploaded file can be archived using the archive_model method of the client object.
The archive_model method takes the following parameters:
- model_id: The id of the model to archive
- archive: An Archive object that contains the reason for archiving the model

This is useful for cleaning up the files if you are going to do multiple runs of the notebook.

In [16]:
archive_reason = istari_digital.Archive(
    reason="This file was used for an example"
)

archived_model = client.archive_model(
    model_id=model_id,
    archive=archive_reason,
)

assert archived_model is not None
assert archived_model.archive_status.name == istari_digital.ArchiveStatusName.ARCHIVED